In [1]:
import pandas as pd
import importlib
from datetime import datetime
from dotenv import load_dotenv
from tqdm.notebook import tqdm

from langsmith import Client
from langchain.smith.evaluation import RunEvalConfig
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.checkpoint.sqlite import SqliteSaver

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


In [2]:
import importlib
from langgraph.checkpoint.sqlite import SqliteSaver # <--- Importamos o SqliteSaver

# --- 1. SELECIONE O WORKFLOW PARA TESTAR ---
WORKFLOW_UNDER_TEST = "boleta_trader"

# --- 2. PREPARA A MEMÓRIA (CHECKPOINTER) ---
# Usamos um banco de dados SQLite em memória para persistir o estado da conversa durante a execução.
memory = SqliteSaver.from_conn_string(":memory:")

# --- 3. CARREGAMENTO DINÂMICO DO "PLUG-IN" ---
print(f"Carregando o workflow: '{WORKFLOW_UNDER_TEST}'...")
try:
    workflow_module_path = f"workflows.{WORKFLOW_UNDER_TEST}.workflow"
    evaluator_module_path = f"workflows.{WORKFLOW_UNDER_TEST}.evaluator"

    workflow_module = importlib.import_module(workflow_module_path)
    evaluator_module = importlib.import_module(evaluator_module_path)

    # 👇 A CORREÇÃO ESTÁ AQUI: Injetamos a memória no workflow
    local_app = workflow_module.get_workflow(checkpointer=memory)
    universal_evaluator = evaluator_module.get_evaluator()

    DATASET_CSV_PATH = f"workflows/{WORKFLOW_UNDER_TEST}/data/dataset.csv"
    print("✅ Workflow e avaliador carregados com sucesso (com persistência de estado).")

except ModuleNotFoundError as e:
    print(f"❌ ERRO: Workflow '{WORKFLOW_UNDER_TEST}' não encontrado. Detalhe: {e}")


# --- 4. CONFIGURAÇÕES ADICIONAIS ---
API_URL = "http://localhost:8000/api/v1/atendente" # Para o modo 'api'
load_dotenv() # Carrega chaves do .env local
PROJECT_NAME_PREFIX = "vizu-workflow-eval"

Carregando o workflow: 'boleta_trader'...
✅ Workflow e avaliador carregados com sucesso (com persistência de estado).


In [3]:
# Configura o cliente LangSmith (apenas para tracing, não para avaliação automática)
client = Client()
project_name = f"{PROJECT_NAME_PREFIX}-{WORKFLOW_UNDER_TEST}-{datetime.now().strftime('%Y%m%d-%H%M')}"

# 1. Carrega o dataset de teste
eval_df = pd.read_csv(DATASET_CSV_PATH)
print(f"Dataset com {len(eval_df)} passos de conversação carregado.")

# 2. Prepara a Simulação
# Usamos um 'checkpointer' para manter a memória do grafo entre as chamadas
memory = SqliteSaver.from_conn_string(":memory:")
# Compilamos o grafo com a memória
local_app_with_memory = workflow_module.get_workflow(checkpointer=memory)

# O thread_id garante que todas as invocações façam parte da mesma conversa
config = {"configurable": {"thread_id": "conversa_unica_de_teste"}}

# Lista para armazenar os resultados de cada passo
results = []

print(f"\n--- INICIANDO SIMULAÇÃO SEQUENCIAL DA CONVERSA ---")
print(f"Projeto no LangSmith: {project_name}\n")

# 3. O Iterador: Executa o workflow para cada mensagem, em sequência
for index, row in tqdm(eval_df.iterrows(), total=eval_df.shape[0], desc="Processando Mensagens"):
    
    # Prepara o input para o grafo
    human_message = HumanMessage(content=row['message'], name=str(row['phone_number']))
    inputs = {"messages": [human_message]}
    
    # Invoca o grafo. O checkpointer garante a persistência do estado.
    final_state = None
    try:
        # Usamos o 'stream' para ter mais controle e observabilidade
        for chunk in local_app_with_memory.stream(inputs, config, stream_mode="values"):
            final_state = chunk
        
        # Armazena o resultado bem-sucedido
        results.append({
            "test_id": row['test_id'],
            "input_message": row['message'],
            "final_state": final_state,
            "golden_answer": row['golden_answer'],
            "status": "✅ SUCESSO"
        })
        print(f"\n[Passo {index+1}] Mensagem '{row['message']}' processada.")
        if final_state.get('boleta_extraida'):
            print(f"  └── ✨ Boleta Gerada: {final_state['boleta_extraida']}")

    except Exception as e:
        # Captura qualquer erro que ocorra durante a invocação
        results.append({
            "test_id": row['test_id'],
            "input_message": row['message'],
            "final_state": None,
            "golden_answer": row['golden_answer'],
            "status": f"❌ FALHA: {type(e).__name__}"
        })
        print(f"\n[Passo {index+1}] FALHA ao processar mensagem '{row['message']}': {e}")


print(f"\n--- ✅ SIMULAÇÃO CONCLUÍDA ---")
# Agora, você pode inspecionar os traces individuais no projeto LangSmith.
# O relatório de resultados na Célula 4 ainda funcionará, mas com os dados que coletamos aqui.

Dataset com 15 passos de conversação carregado.

--- INICIANDO SIMULAÇÃO SEQUENCIAL DA CONVERSA ---
Projeto no LangSmith: vizu-workflow-eval-boleta_trader-20251007-2151



Processando Mensagens:   0%|          | 0/15 [00:00<?, ?it/s]

--- Nó: Gatekeeper ---

[Passo 1] Mensagem 'Bom dia pessoal, como está o mercado hoje?' processada.
--- Nó: Gatekeeper ---

[Passo 2] Mensagem 'Vendo 20k a 5.15' processada.
--- Nó: Gatekeeper ---

[Passo 3] Mensagem 'Pago 5.145 nos 20k' processada.
--- Nó: Gatekeeper ---

[Passo 4] Mensagem 'Ok, pode ser. Trava pra mim.' processada.
--- Nó: Gatekeeper ---

[Passo 5] Mensagem 'Alguém viu o jogo ontem?' processada.
--- Nó: Gatekeeper ---
--- Nó: Extrator de Boleta ---
>> Boleta extraída com sucesso: {'vendedor': 'John Smith', 'comprador': 'Jane Doe', 'valor_cotacao': 12.99, 'valor_total': 25.98}
--- Nó: Formatador de Boleta ---
>> Mensagem final formatada:
✅ **BOLETA DE CONFIRMAÇÃO** ✅
**Vendedor:** John Smith
**Comprador:** Jane Doe
**Cotação:** R$ 12.990
**Volume:** $25.98

[Passo 6] Mensagem 'Fechado! pode mandar' processada.
  └── ✨ Boleta Gerada: {'vendedor': 'John Smith', 'comprador': 'Jane Doe', 'valor_cotacao': 12.99, 'valor_total': 25.98}
--- Nó: Gatekeeper ---
--- Nó: Extrator

In [ ]:
# Puxa os resultados do LangSmith para análise
results_df = client.get_run_results(run_id=str(run.id))

# Limpa e formata o DataFrame para melhor visualização
results_df['feedback_score'] = results_df['feedback.boleta_match_score'].fillna(0).astype(int)
results_df['input_message'] = results_df['input.message']
results_df['feedback_comment'] = results_df['feedback.comment']

# Calcula as métricas de sucesso
total_testes = len(results_df)
sucessos = results_df['feedback_score'].sum()
taxa_sucesso = (sucessos / total_testes) * 100 if total_testes > 0 else 0

print(f"--- 📈 Relatório de Avaliação: {project_name} ---")
print(f"Total de Passos de Conversa Avaliados: {total_testes}")
print(f"Sucessos (Golden Answer Correta): {sucessos}")
print(f"Taxa de Sucesso: {taxa_sucesso:.2f}%")

# Exibe os casos de falha para depuração rápida
falhas_df = results_df[results_df['feedback_score'] == 0]
if not falhas_df.empty:
    print("\n--- ❌ Casos com Falha ---")
    display(falhas_df[['input_message', 'feedback_comment']].reset_index(drop=True))
else:
    print("\n✅ Todos os testes passaram com sucesso!")

print(f"\n🔗 Link para o projeto no LangSmith: {run.url}")

AttributeError: 'Client' object has no attribute 'get_run_results'